# Level 1 — Problem Framing

## Water Resource Management for Kenyan Agriculture

This notebook frames the core problem: predicting soil moisture, evapotranspiration (ET), and optimal irrigation schedules for drought-prone regions in Kenya.

### Problem Statement
- **Goal**: Develop a water balance model to predict soil moisture deficits and guide irrigation decisions
- **Challenge**: Limited water resources in arid/semi-arid Kenya; need intelligent, data-driven irrigation
- **Success Metric**: Reduce irrigation water use by 20% while maintaining crop yield

### Key Variables
- Soil moisture (mm): Current water stored in soil
- Rainfall (mm/day): Precipitation input
- Evapotranspiration (mm/day): Water lost to atmosphere
- Irrigation (mm/day): Controlled water addition
- Drainage (mm/day): Water loss to deep soil layers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set up paths
repo_root = Path('.').resolve().parent
data_dir = repo_root / 'data' / 'raw'

print(f"Repository root: {repo_root}")
print(f"Data directory: {data_dir}")
print(f"Data directory exists: {data_dir.exists()}")

# List available CSV files
csv_files = list(data_dir.glob('*.csv'))
print(f"\nAvailable CSV files: {len(csv_files)}")
for f in csv_files:
    print(f"  - {f.name}")

In [ ]:
# Load weather data
weather_file = data_dir / 'weather_daily.csv'
if weather_file.exists():
    weather = pd.read_csv(weather_file)
    print("Weather Data Summary:")
    print(f"Shape: {weather.shape}")
    print(f"Columns: {list(weather.columns)}")
    print(f"\nFirst 5 rows:")
    display(weather.head())
    print(f"\nStatistics:")
    display(weather.describe())
else:
    print(f"Weather file not found at {weather_file}")

In [ ]:
# Load soil sensor data
soil_file = data_dir / 'soil_sensor_data.csv'
if soil_file.exists():
    soil = pd.read_csv(soil_file)
    print("Soil Sensor Data Summary:")
    print(f"Shape: {soil.shape}")
    print(f"Columns: {list(soil.columns)}")
    print(f"\nFirst 5 rows:")
    display(soil.head())
    print(f"\nZones: {soil['Zone'].unique()}")
else:
    print(f"Soil file not found at {soil_file}")

In [ ]:
# Load crop parameters
crop_file = data_dir / 'crop_zone_parameters.csv'
if crop_file.exists():
    crops = pd.read_csv(crop_file)
    print("Crop Zone Parameters:")
    print(f"Shape: {crops.shape}")
    print(f"Columns: {list(crops.columns)}")
    print(f"\nData:")
    display(crops)
else:
    print(f"Crop file not found at {crop_file}")

In [ ]:
# Data Dictionary
data_dict = {
    'weather_daily.csv': {
        'Date': 'Calendar date (YYYY-MM-DD)',
        'Rainfall_mm': 'Daily precipitation (mm)',
        'Tmax_C': 'Maximum air temperature (Celsius)',
        'Tmin_C': 'Minimum air temperature (Celsius)',
        'Humidity_%': 'Relative humidity (%)',
        'WindSpeed_ms': 'Wind speed (m/s)',
        'SolarRadiation_MJm2': 'Solar radiation (MJ/m²/day)'
    },
    'soil_sensor_data.csv': {
        'Date': 'Calendar date (YYYY-MM-DD)',
        'Zone': 'Irrigation zone (Zone_A, Zone_B, Zone_C)',
        'SoilMoisture_mm': 'Current soil water content (mm)',
        'Tank_Level_m': 'Water tank depth (meters)',
        'Pump_Flow_mm': 'Irrigation flow rate (mm/day)'
    },
    'crop_zone_parameters.csv': {
        'Zone': 'Irrigation zone identifier',
        'Area_ha': 'Crop area (hectares)',
        'Crop_Type': 'Crop grown in zone',
        'Kc': 'Crop coefficient (0-1.3)',
        'MAD_%': 'Maximum Allowable Depletion (%)',
        'FC_mm': 'Field Capacity (mm)',
        'PWP_mm': 'Permanent Wilting Point (mm)'
    }
}

print("\n=== DATA DICTIONARY ===")
for file_name, variables in data_dict.items():
    print(f"\n{file_name}:")
    for var, desc in variables.items():
        print(f"  {var}: {desc}")

In [ ]:
# Core Function: Estimate Evapotranspiration (ET)
def estimate_et(tmax, tmin, humidity, wind, solar_rad, kc=0.8):
    """
    Estimate daily evapotranspiration using simplified Hargreaves-Samani formula.
    
    Parameters:
    -----------
    tmax, tmin : float
        Maximum and minimum temperature (°C)
    humidity : float
        Relative humidity (%)
    wind : float
        Wind speed (m/s)
    solar_rad : float
        Solar radiation (MJ/m²/day)
    kc : float
        Crop coefficient (0-1.3)
    
    Returns:
    --------
    et_mm : float
        Evapotranspiration (mm/day)
    
    Notes:
    ------
    Simplified formula: ET = 0.0023 * Ra * (Tmean + 17.8) * sqrt(Tmax - Tmin) * Kc
    where Ra is converted from solar radiation.
    """
    t_mean = (tmax + tmin) / 2.0
    t_range = tmax - tmin
    
    # Hargreaves constant and formula
    et_temp = 0.0023 * solar_rad * (t_mean + 17.8) * np.sqrt(max(t_range, 0.1)) * kc
    
    # Apply humidity adjustment (reduce ET if very humid)
    humidity_factor = 1.0 - (humidity - 50) / 100 if humidity > 50 else 1.0
    et_mm = et_temp * max(humidity_factor, 0.5)
    
    return max(et_mm, 0.0)

# Test the function
test_et = estimate_et(tmax=28, tmin=18, humidity=70, wind=2, solar_rad=20, kc=0.8)
print(f"Test ET calculation: {test_et:.2f} mm/day")

In [ ]:
# Core Function: Daily Water Balance
def daily_water_balance(s_prev, rainfall, et, drainage, irrigation):
    """
    Compute daily soil moisture using water balance equation.
    
    S(t+1) = S(t) + P(t) - ET(t) - D(t) + I(t)
    
    Parameters:
    -----------
    s_prev : float
        Soil moisture at time t (mm)
    rainfall : float
        Daily precipitation (mm/day)
    et : float
        Daily evapotranspiration (mm/day)
    drainage : float
        Deep drainage loss (mm/day)
    irrigation : float
        Applied irrigation (mm/day)
    
    Returns:
    --------
    s_new : float
        Soil moisture at time t+1 (mm)
    """
    s_new = s_prev + rainfall - et - drainage + irrigation
    return max(s_new, 0.0)

# Test the function
s_test = daily_water_balance(s_prev=150, rainfall=5, et=4, drainage=1, irrigation=0)
print(f"Test water balance: S(t+1) = {s_test:.2f} mm")

In [ ]:
# Visualization: Weather Time Series
if weather_file.exists():
    weather['Date'] = pd.to_datetime(weather['Date'])
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle('Weather Data Overview', fontsize=16, fontweight='bold')
    
    # Rainfall
    axes[0, 0].bar(weather['Date'], weather['Rainfall_mm'], color='steelblue', alpha=0.7)
    axes[0, 0].set_title('Daily Rainfall')
    axes[0, 0].set_ylabel('Rainfall (mm/day)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Temperature range
    axes[0, 1].fill_between(weather['Date'], weather['Tmin_C'], weather['Tmax_C'], alpha=0.5, label='Temp Range')
    axes[0, 1].plot(weather['Date'], (weather['Tmax_C'] + weather['Tmin_C']) / 2, 'r-', label='Mean Temp')
    axes[0, 1].set_title('Temperature Dynamics')
    axes[0, 1].set_ylabel('Temperature (°C)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Humidity
    axes[1, 0].plot(weather['Date'], weather['Humidity_%'], color='green', linewidth=2)
    axes[1, 0].set_title('Relative Humidity')
    axes[1, 0].set_ylabel('Humidity (%)')
    axes[1, 0].set_ylim([0, 100])
    axes[1, 0].grid(True, alpha=0.3)
    
    # Solar Radiation
    axes[1, 1].plot(weather['Date'], weather['SolarRadiation_MJm2'], color='orange', linewidth=2)
    axes[1, 1].set_title('Solar Radiation')
    axes[1, 1].set_ylabel('Solar Rad (MJ/m²/day)')
    axes[1, 1].grid(True, alpha=0.3)
    
    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualization: Soil Moisture by Zone
if soil_file.exists():
    soil['Date'] = pd.to_datetime(soil['Date'])
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for zone in soil['Zone'].unique():
        zone_data = soil[soil['Zone'] == zone]
        ax.plot(zone_data['Date'], zone_data['SoilMoisture_mm'], marker='o', label=zone, linewidth=2, markersize=4)
    
    ax.set_title('Soil Moisture Dynamics by Zone', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Soil Moisture (mm)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Summary

**Level 1 Deliverables Completed:**
1. ✓ Identified problem: Water resource management for Kenyan agriculture
2. ✓ Loaded and inspected all three datasets
3. ✓ Created data dictionary with variable definitions
4. ✓ Implemented ET estimation function (Hargreaves-Samani)
5. ✓ Implemented daily water balance model
6. ✓ Generated weather and soil moisture visualizations

**Key Findings:**
- Multiple irrigation zones (A, B, C) with varying crop types and soil properties
- Weather shows seasonal patterns; rainfall is sparse (typical of arid Kenya)
- Soil moisture varies significantly by zone, indicating zone-specific irrigation needs

**Next Steps (Level 2):**
- Vectorize ET and water balance calculations for performance
- Analyze floating-point precision errors in iterative simulations
- Quantify impact of error propagation on irrigation decisions